# exp_ablation — component ablation tables for the writeup

Loads the cached features for a pool (`POOL_TAG`) and runs the LambdaMART on feature-group subsets:
**add-one-in** (each view's incremental contribution) and **leave-one-out** (each view's marginal value),
plus per-view standalone. Tuning is FIXED (num_leaves=15, train=TREC21+KZ, TREC22 eval) so only the feature
set varies — the clean ablation. CPU only (features cached). Run with POOL_TAG='R' and 'nqs' for the pool ablation.


## Setup (Colab or local — CPU is fine)


In [ ]:
!pip install -q git+https://github.com/semajyllek/ctmatch.git
!pip install -q lightgbm pytrec_eval datasets pandas tqdm


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
import os, json
os.environ['HF_HUB_DISABLE_XET'] = '1'; os.environ['HF_HOME'] = '/content/hf_cache'
DATA_ROOT = '/content/drive/MyDrive/ct_data23'
import numpy as np, pandas as pd, lightgbm as lgb
from datasets import load_dataset
from ctmatch.experiments import ExperimentConfig, load_eval, pytrec_metrics
POOL_TAG = 'R'   # 'R' or 'nqs'
cfg = ExperimentConfig(data_root=DATA_ROOT, pool_tag=POOL_TAG)
print('pool:', cfg.pool_path())


In [ ]:
# Load pool + all cached features (whatever exists for this pool).
corpus_ids = {r['text'].strip() for r in load_dataset(cfg.index2docid_hf, data_files='index2docid.txt', split='train')}
sets = load_eval(cfg, ['trec21','kz','trec22'])
pool = json.load(open(cfg.pool_path()))
def load_jsonl(path):
    d={};
    for l in open(path): r=json.loads(l); d[(r['source'],r['topic_id'],r['doc_id'])]=r
    return d
rfeat = load_jsonl(cfg.feat_file('retrieval_feats'))
llm = {k:v['llm_score'] for k,v in load_jsonl(cfg.feat_file('llm_scores')).items()}
topi = {k:v['topicality'] for k,v in load_jsonl(cfg.feat_file('topicality')).items()} if os.path.exists(cfg.feat_file('topicality')) else None
cm = {k:v['condition_match'] for k,v in load_jsonl(cfg.feat_file('condition_match')).items()} if os.path.exists(cfg.feat_file('condition_match')) else None
z = np.load(cfg.ce_cache_path('clf_R'), allow_pickle=True); clf_f = z['d'].item()
clft_f = np.load(cfg.ce_cache_path('clf_topic'), allow_pickle=True)['d'].item() if os.path.exists(cfg.ce_cache_path('clf_topic')) else None
print('features loaded | clf_topic:', clft_f is not None, '| topicality:', topi is not None, '| condition_match:', cm is not None)


In [ ]:
# Feature groups (only those available). Each group = a 'view'.
GROUPS = {'retrieval': ['bm25','bm25_rank','dense','dense_rank','rrf'], 'clf_R': ['clf_rel','clf_partial']}
if clft_f is not None: GROUPS['clf_topic'] = ['clf_topic_rel','clf_topic_partial']
GROUPS['judge'] = ['llm_yesno']
if topi is not None: GROUPS['topic_judge'] = ['topicality']
if cm is not None: GROUPS['cond_match'] = ['condition_match']
ALLF = [f for g in GROUPS.values() for f in g]
def fd(s,t,d):
    rf = rfeat.get((s,t,d),{}); cr,cp = clf_f.get((s,t,d),(0.,0.))
    v = {'bm25':rf.get('bm25',0.),'bm25_rank':rf.get('bm25_rank',cfg.cand_k),'dense':rf.get('dense',0.),
         'dense_rank':rf.get('dense_rank',cfg.cand_k),'rrf':rf.get('rrf',0.),'clf_rel':cr,'clf_partial':cp,
         'llm_yesno':llm.get((s,t,d),cfg.llm_floor)}
    if clft_f is not None: tr,tp=clft_f.get((s,t,d),(0.,0.)); v['clf_topic_rel']=tr; v['clf_topic_partial']=tp
    if topi is not None: v['topicality']=topi.get((s,t,d),0.)
    if cm is not None: v['condition_match']=cm.get((s,t,d),0.)
    return v
def build(s):
    rows,y,g = [],[],[]; rel=sets[s]['rel_dict']
    for t,docs in pool[s].items():
        docs=[d for d in docs if d in corpus_ids]; g.append(len(docs))
        for d in docs: rows.append(fd(s,t,d)); y.append(int(rel[t].get(d,0)))
    return rows,np.array(y),g
tr_rows,ytr_a,gtr_a=build('trec21'); kz_rows,ykz,gkz=build('kz'); te_rows,yte,gte=build('trec22')
tr_rows=tr_rows+kz_rows; ytr=np.concatenate([ytr_a,ykz]); gtr=gtr_a+gkz
def X(rows,names): return np.array([[r[n] for n in names] for r in rows],dtype=np.float32)
te_meta=[(t,d) for t in pool['trec22'] for d in pool['trec22'][t] if d in corpus_ids]


In [ ]:
# Fixed tuning; vary only the feature set. Returns TREC22 NDCG@10 / P@10 / MRR.
def evalset(names):
    b=lgb.train({'objective':'lambdarank','metric':'ndcg','ndcg_eval_at':[10],'num_leaves':15,
                 'min_data_in_leaf':20,'learning_rate':0.05,'lambda_l2':1.0,'verbose':-1},
                lgb.Dataset(X(tr_rows,names),ytr,group=gtr),num_boost_round=50)
    p=b.predict(X(te_rows,names)); run={}; i=0
    for (t,d) in te_meta: run.setdefault(t,{})[d]=float(p[i]); i+=1
    q={t:{d:int(r) for d,r in sets['trec22']['rel_dict'][t].items()} for t in run}
    return pytrec_metrics(run,q,10)


### Add-one-in — each view's incremental contribution (cumulative)


In [ ]:
order = [g for g in ['retrieval','clf_R','clf_topic','judge','topic_judge','cond_match'] if g in GROUPS]
rows=[]; cum=[]
for g in order:
    cum = cum + GROUPS[g]; m = evalset(cum)
    rows.append({'+ view': g, 'features': len(cum), **m})
add_df = pd.DataFrame(rows); add_df


### Leave-one-out — each view's marginal value (drop from full)


In [ ]:
full = evalset(ALLF)
rows=[{'dropped':'(none = full)', **full, 'delta':0.0}]
for g in order:
    names=[f for f in ALLF if f not in GROUPS[g]]; m=evalset(names)
    rows.append({'dropped':g, **m, 'delta':round(m['ndcg@10']-full['ndcg@10'],4)})
loo_df = pd.DataFrame(rows); loo_df


### Per-view standalone (rank TREC22 by that view alone, in the ensemble)


In [ ]:
rows=[{'view':g, **evalset(GROUPS[g])} for g in order]
pd.DataFrame(rows).sort_values('ndcg@10', ascending=False)


## For the writeup
- **Add-one-in** is the headline ablation: the NDCG@10 ladder as each view is added (retrieval → clf_R →
  clf_topic → judge → topicality → condition_match). **Leave-one-out** gives each view's marginal value.
- Run this with `POOL_TAG='R'` and `POOL_TAG='nqs'` and put both columns side by side = the **pool ablation**
  (does NQS retrieval lift the whole ladder?). Plus the §2h representation ablation (elig_first vs head etc.)
  and the progression table = a complete ablation section.
